In [0]:
%run ./watermark_utils

In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

# COMMAND ----------
import time
import requests
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, TimestampType, DoubleType, LongType, StringType
)

PRICE_SCHEMA = StructType([
    StructField("date", TimestampType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("symbol", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
])


def clean_price_frame(df_symbol: pd.DataFrame, symbol: str) -> pd.DataFrame:
    if isinstance(df_symbol.columns, pd.MultiIndex):
        df_symbol.columns = df_symbol.columns.get_level_values(0)
    if "Adj Close" in df_symbol.columns:
        df_symbol["Close"] = df_symbol["Adj Close"]
    df_symbol.drop(columns=["Adj Close"], errors="ignore", inplace=True)
    df_symbol.rename(columns={"Date": "date"}, inplace=True)
    df_symbol["date"] = pd.to_datetime(df_symbol["date"])
    df_symbol["symbol"] = symbol
    df_symbol["ingestion_timestamp"] = datetime.now()
    df_symbol.columns = (
        df_symbol.columns.str.strip().str.lower()
        .str.replace(" ", "_").str.replace("-", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )
    return df_symbol


def fetch_prices_incremental(
    spark, symbol_list, bronze_table_full_name, failed_table_full_name,
    watermark_table_full_name, watermark_source, batch_size, default_start,
    group_by_ticker=True,
):
    max_date_dict = get_watermark_dict(spark, watermark_table_full_name, watermark_source)
    end_date = datetime.today().strftime("%Y-%m-%d")

    symbol_start_dates = {}
    for s in symbol_list:
        last_date = max_date_dict.get(s)
        symbol_start_dates[s] = (
            (pd.to_datetime(last_date) + timedelta(days=1)).strftime("%Y-%m-%d")
            if last_date else default_start
        )

    invalid_symbols = []

    for i in range(0, len(symbol_list), batch_size):
        batch = symbol_list[i:i + batch_size]
        start_date = min(symbol_start_dates[s] for s in batch)
        print(f"batch {i // batch_size + 1} ({len(batch)} symbols, from {start_date})")

        try:
            data = yf.download(
                batch, start=start_date, end=end_date,
                threads=True, group_by="ticker" if group_by_ticker else None,
                progress=False,
            )

            all_data = []
            for s in batch:
                if group_by_ticker and s not in data.columns.levels[0]:
                    invalid_symbols.append(s)
                    continue
                df_symbol = (data[s] if group_by_ticker else data).reset_index()
                df_symbol = clean_price_frame(df_symbol, s)
                df_symbol = df_symbol[df_symbol["date"] >= symbol_start_dates[s]]
                if "close" in df_symbol.columns:
                    df_symbol = df_symbol.dropna(subset=["close"])
                if not df_symbol.empty:
                    all_data.append(df_symbol)

            if all_data:
                combined_df = pd.concat(all_data)
                combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

                spark.createDataFrame(combined_df, schema=PRICE_SCHEMA) \
                    .write.format("delta").mode("append") \
                    .partitionBy("symbol") \
                    .saveAsTable(bronze_table_full_name)

                batch_max_dates = (
                    combined_df.groupby("symbol")["date"].max().dt.date.to_dict()
                )
                update_watermark(spark, watermark_table_full_name, watermark_source, batch_max_dates)

        except Exception as e:
            print(f"Error fetching batch {batch}: {e}")

        time.sleep(1)

    if invalid_symbols:
        failed_df = spark.createDataFrame([(s, datetime.now()) for s in invalid_symbols],
                                           ["symbol", "failed_at"])
        failed_df.write.format("delta").mode("append").saveAsTable(failed_table_full_name)
        print(f"{len(invalid_symbols)} symbols failed — see {failed_table_full_name}")

    print("Batch processing completed")


def fetch_mfapi_nav_incremental(
    spark, scheme_list, bronze_table_full_name, watermark_table_full_name,
    watermark_source, mfapi_base_url, batch_size, max_workers,
):
    from concurrent.futures import ThreadPoolExecutor, as_completed

    max_date_dict = get_watermark_dict(spark, watermark_table_full_name, watermark_source)

    for i in range(0, len(scheme_list), batch_size):
        batch = scheme_list[i:i + batch_size]
        all_nav_rows = []
        print(f"Processing batch {i // batch_size + 1}")

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(requests.get, f"{mfapi_base_url}/{code}", timeout=60): code
                for code in batch
            }
            for future in as_completed(futures):
                scheme_code = futures[future]
                try:
                    response = future.result()
                    if response.status_code != 200:
                        continue
                    nav_data = response.json().get("data", [])
                    if not nav_data:
                        continue
                    max_date = max_date_dict.get(scheme_code)
                    for row in nav_data:
                        row_date = datetime.strptime(row["date"], "%d-%m-%Y").date()
                        if max_date and row_date <= max_date:
                            continue
                        row["schemeCode"] = scheme_code
                        row["date"] = row_date
                        all_nav_rows.append(row)
                except Exception as e:
                    print(f"Error fetching data for scheme code {scheme_code}: {e}")

        if all_nav_rows:
            nav_df = spark.createDataFrame(all_nav_rows) \
                .withColumn("ingestion_timestamp", F.current_timestamp()) \
                .withColumn("nav", F.col("nav").cast("double"))
            nav_df.write.format("delta").mode("append") \
                .partitionBy("schemeCode").saveAsTable(bronze_table_full_name)

            batch_max_dates = {}
            for row in all_nav_rows:
                sc, d = row["schemeCode"], row["date"]
                if sc not in batch_max_dates or d > batch_max_dates[sc]:
                    batch_max_dates[sc] = d
            update_watermark(spark, watermark_table_full_name, watermark_source, batch_max_dates)

        time.sleep(1)

    print("Full historical/incremental NAV load completed")